# QAX-Smart (`qaxs`) fine-tuning — Qwen2.5-Coder-0.5B → LoRA → GGUF

Run on a Colab **T4 GPU runtime** (Runtime → Change runtime type → T4 GPU).

This notebook:
1. Installs Unsloth (fast QLoRA finetuning) + llama.cpp conversion tooling.
2. Loads the `qaxs` training dataset (generated by `training/generate_dataset.py`,
   built from the verified QAX v3.0.0 syntax reference — see `HANDOFF.md`).
3. Fine-tunes Qwen2.5-Coder-0.5B-Instruct with LoRA.
4. Evaluates base vs. fine-tuned on the held-out eval split using the
   *same* deterministic checks `qaxs` itself runs at inference time
   (shell-syntax sanity + semantic-intent rules), so "did the fine-tune
   help" is measured the same way the product measures correctness.
5. Merges the LoRA adapter into the base weights and exports GGUF Q8_0,
   ready to drop into `models/qaxs-qwen2.5-coder-0.5b-lora-merged-q8_0.gguf`.

**Before you ship anything**: re-run section 6 (eval) after training and
actually look at the failures, not just the pass rate number. A 0.5B
model fine-tuned on a few hundred synthetic examples will still get
things wrong — the point of this notebook is to measure how *much*
better than the base model it is, not to promise perfection.

## 1. Install dependencies

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade transformers peft trl accelerate bitsandbytes datasets
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
# NOTE: llama.cpp's old Makefile build was removed and replaced entirely by
# CMake (verified directly - the Makefile now just prints an error telling
# you to use CMake instead). GGML_CUDA=OFF is fine here since quantizing
# doesn't need GPU acceleration - it's a one-off CPU-bound pass over the
# merged weights.
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=OFF 2>&1 | tail -5
!cd /content/llama.cpp && cmake --build build --config Release --target llama-quantize -j 2>&1 | tail -10


## 2. Get the dataset

Either upload `qaxs_dataset.jsonl` (and `qaxs_dataset.eval.jsonl`) from
your local `training/` folder, or regenerate it here from scratch — the
generator has no external dependencies beyond the Python standard library.

In [ ]:
import os

USE_UPLOAD = False  # flip to True to upload files.jsonl manually instead

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()  # select qaxs_dataset.jsonl and qaxs_dataset.eval.jsonl
else:
    # Regenerate in-place. Paste generate_dataset.py's content here if you
    # haven't uploaded the repo — simplest is to upload just that one file.
    from google.colab import files
    print("Upload generate_dataset.py from the qaxs repo's training/ folder:")
    up = files.upload()
    assert "generate_dataset.py" in up, "expected generate_dataset.py"
    with open("generate_dataset.py", "wb") as f:
        f.write(up["generate_dataset.py"])
    !python3 generate_dataset.py --out qaxs_dataset.jsonl --per-template 8

print(os.popen("wc -l qaxs_dataset.jsonl qaxs_dataset.eval.jsonl").read())


## 3. Format into the exact prompt qaxs uses at inference time

This mirrors `src/core/prompt.ts`'s `BASE_PROMPT` / `buildQaxPrompt` — the
model needs to learn to respond correctly to the SAME prompt shape qaxs
will actually send it, not some other instruction format.

In [ ]:
import json

QAX_SYNTAX_HINT = """
QAX SHELL: use $VAR, $(cmd), $((expr)) for expansion/substitution/arithmetic; test/[ ] for conditions; if/while/for for control flow; built-ins like ls, cat, cp, mv, rm, mkdir, touch, cd, pwd.

EXAMPLES (format only - use the real environment values above for any actual paths):
Request: check if notes.txt exists
Command: [ -f notes.txt ]

Request: create a folder called build
Command: mkdir -p build

Request: list files sorted by size
Command: ls -lhS
"""

def build_prompt(row):
    base = f"""You are QAX-Smart\'s command generation engine.

Your ONLY task is to convert the user\'s natural-language request into exactly ONE executable command for the specified shell.

OUTPUT ONLY THE COMMAND.

DO NOT:
- explain anything
- greet the user
- use Markdown
- use code fences
- provide labels
- provide multiple commands
- provide alternatives
- describe the command
- output conversational text

TARGET SHELL: {row["shell"]}
OPERATING SYSTEM: {row["os"]}

USER REQUEST:
{row["request"]}"""
    if row["shell"] == "qax":
        base = base.replace("USER REQUEST:", QAX_SYNTAX_HINT + "\nUSER REQUEST:")
    return base

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_jsonl("qaxs_dataset.jsonl")
eval_rows = load_jsonl("qaxs_dataset.eval.jsonl")

for row in train_rows[:2]:
    print(build_prompt(row))
    print("---> ", row["command"])
    print("=====")


## 4. Load the base model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 512  # prompts + commands here are short; keep this small for speed

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-0.5B-Instruct",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=7,
)


## 5. Build the SFT dataset and train

In [ ]:
from datasets import Dataset

def to_example(row):
    prompt = build_prompt(row)
    # Qwen chat template wraps this; we train the model to emit ONLY the
    # command as its full assistant turn, matching what the normalizer/
    # extractor expect at inference time.
    text = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": row["command"]},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = Dataset.from_list([to_example(r) for r in train_rows])
print(train_ds[0]["text"][:800])


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=7,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()


## 6. Evaluate: base vs. fine-tuned, using qaxs's own correctness rules

This ports the same deterministic semantic checks from
`src/core/semantic-validator.ts` into Python, so "pass" here means the
same thing "pass" means inside qaxs at actual runtime — not just string
similarity to the reference command.

In [ ]:
import re

# NOTE: these rules mirror src/core/semantic-validator.ts in the qaxs repo
# EXACTLY, including a regex-precedence bug fix found via that project's
# own regression test suite: `\b(create|make)\b.*\bfolder|directory\b`
# (missing parens around folder|directory) would match ANY request
# containing the word "directory" regardless of a create/make verb, due
# to `|` having lower precedence than expected. If you ever edit these
# rules, mirror the change back into semantic-validator.ts too - keeping
# eval-time correctness checks and runtime correctness checks in sync is
# the entire point of this cell.
SEMANTIC_RULES = [
    (re.compile(r"\bpdfs?\b", re.I), re.compile(r"\.pdf|pdf", re.I)),
    (re.compile(r"\b(recursive|recursively|all subfolders|everywhere)\b", re.I),
     re.compile(r"(-r\b|-R\b|--recurse|-Recurse|/s\b|find\s)", re.I)),
    (re.compile(r"\bprocess(es)?\b", re.I), re.compile(r"(ps\b|tasklist|Get-Process|pgrep)", re.I)),
    (re.compile(r"\bport\b", re.I), re.compile(r"(netstat|lsof|Get-NetTCPConnection|ss\s)", re.I)),
    (re.compile(r"\b(create|make)\b.*\b(folder|directory)\b", re.I), re.compile(r"(mkdir|md\b|New-Item)", re.I)),
    (re.compile(r"\b(delete|remove|erase)\b", re.I), re.compile(r"(rm\b|del\b|Remove-Item|rmdir|rd\b)", re.I)),
]

def semantic_pass(request, command):
    for req_pat, cmd_pat in SEMANTIC_RULES:
        if req_pat.search(request) and not cmd_pat.search(command):
            return False
    return True

def generate(model, tokenizer, row, max_new_tokens=48):
    prompt = build_prompt(row)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.1)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return text.strip().split("\n")[0].strip()

FastLanguageModel.for_inference(model)

results = []
for row in eval_rows:
    generated = generate(model, tokenizer, row)
    passed = semantic_pass(row["request"], generated)
    results.append({"shell": row["shell"], "request": row["request"], "expected": row["command"], "generated": generated, "semantic_pass": passed})

num_passed = sum(r["semantic_pass"] for r in results)
pass_rate = num_passed / len(results)
print(f"Fine-tuned semantic pass rate on held-out eval set: {pass_rate:.1%} ({num_passed}/{len(results)})")

print("\nFailures (inspect these before shipping):")
for r in results:
    if not r["semantic_pass"]:
        shell = r["shell"]
        request = r["request"]
        generated = r["generated"]
        expected = r["expected"]
        print(f"  [{shell}] {request!r} -> {generated!r}  (expected: {expected!r})")


### Optional: compare against the un-fine-tuned base model

Re-run generation with the base weights (LoRA disabled) on the same eval
set to get a real before/after number, not just an absolute pass rate.

In [ ]:
with model.disable_adapter():
    base_results = []
    for row in eval_rows:
        generated = generate(model, tokenizer, row)
        base_results.append({"request": row["request"], "semantic_pass": semantic_pass(row["request"], generated)})

base_pass_rate = sum(r["semantic_pass"] for r in base_results) / len(base_results)
print(f"Base model semantic pass rate:       {base_pass_rate:.1%}")
print(f"Fine-tuned model semantic pass rate: {pass_rate:.1%}")


## 7. Merge LoRA into base weights and export GGUF

This produces the file `qaxs`'s `ModelManager` looks for automatically:
`qaxs-qwen2.5-coder-0.5b-lora-merged-q8_0.gguf`.

In [ ]:
merged_dir = "qaxs-merged"
model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")


In [ ]:
!python3 /content/llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile qaxs-merged-f16.gguf --outtype f16
!/content/llama.cpp/build/bin/llama-quantize qaxs-merged-f16.gguf qaxs-qwen2.5-coder-0.5b-lora-merged-q8_0.gguf Q8_0


In [ ]:
from google.colab import files
files.download("qaxs-qwen2.5-coder-0.5b-lora-merged-q8_0.gguf")
print("Drop the downloaded file into <qaxs repo>/models/ — qaxs prefers it over the base model automatically.")
